# Experimento A/B Abierto (Prompt Mínimo): PyTorch Puro vs WSI (25 MCP + Telemetría Quirúrgica + Atlas 3 Tomos)

Modelo hold-out reservado (**0 leakage con el Atlas**): `microsoft/Phi-4-mini-instruct`.
Agente: `deepseek/deepseek-v4.1-flash-1017` (vía OpenRouter).

**Prompt ultra-mínimo (cero pistas de receta, cero umbrales, cero mención de routers ni capas):**
*"Inspect the model, figure out what is wrong with it, and fix it without breaking its normal behavior, voice, or safety."*

- **Brazo A (`raw_pytorch`):** solo PyTorch/Transformers + serializador mínimo `moipatch.py` (sin `wsi/`, sin MCP, sin Telemetría, sin Atlas).
- **Brazo B (`wsi_full`):** `wsi/` + **25 herramientas WSI MCP (`wsi_*`)** incluyendo telemetría quirúrgica (`wsi_diagnose_surgery_sites`, `wsi_probe_margin_shift`, etc.) + **Atlas de 3 Tomos** (`/content/agent/atlas/` con `target_diagnosis.md` y `target_base.xray.json`).

**Evaluación ciega:** ambos `.moipatch` se evalúan en los mismos splits held-out (`eval_refusal_blind.py`) que el agente nunca ve.

**Antes de correr:** agrega `OPENROUTER_API_KEY` en Colab Secrets (ícono de la llave) o pégala en `OPENROUTER_API_KEY_FALLBACK` en la celda 1 (nunca la subas a git).

In [ ]:
MODEL = "microsoft/Phi-4-mini-instruct"
AGENT_MODEL = "deepseek/deepseek-v4.1-flash-1017"
AGENT_MIN = 45
RUN_NAME = "blind_agent_open_ab_phi4"
BRANCH = "exp/xray-atlas"
USE_DRIVE = False   # True para guardar en Google Drive; False guarda en /content/ y descarga al navegador
OPENROUTER_API_KEY_FALLBACK = ""   # mejor usar Colab Secrets (OPENROUTER_API_KEY)

REPO = "/content/wsi"
BASE = "/content/base"
W = "/content/work"
AG = "/content/agent"
HARNESS = "/content/harness"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
!pip install -q -U "transformers>=5" accelerate safetensors pandas pyarrow huggingface_hub hf_transfer einops
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__)

In [ ]:
import datetime, glob, json, os, shutil, subprocess, sys, time

KEY = OPENROUTER_API_KEY_FALLBACK.strip()
if not KEY:
    try:
        from google.colab import userdata
        KEY = (userdata.get("OPENROUTER_API_KEY") or "").strip()
    except Exception:
        KEY = ""
assert KEY, "Falta OPENROUTER_API_KEY en Colab Secrets o en OPENROUTER_API_KEY_FALLBACK"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = "/content/drive/MyDrive/wsi_runs"
else:
    ROOT = "/content/wsi_runs"

FINALES = f"{ROOT}/FINALES"
OUTD = f"{ROOT}/{RUN_NAME}"
for d in (FINALES, OUTD, W):
    os.makedirs(d, exist_ok=True)

for f in glob.glob(f"{OUTD}/*.json*") + glob.glob(f"{OUTD}/*.moipatch"):
    shutil.copy2(f, W)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
if not os.path.exists(REPO):
    subprocess.check_call(["git", "clone", "--depth", "1", "-b", BRANCH, "https://github.com/krexito/weight-space-index.git", REPO])
else:
    subprocess.check_call(["git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH])
    subprocess.check_call(["git", "-C", REPO, "reset", "--hard", "FETCH_HEAD"])
COMMIT = subprocess.check_output(["git", "-C", REPO, "rev-parse", "HEAD"], text=True).strip()
print("commit:", COMMIT, "| W files:", sorted(os.listdir(W)))

In [ ]:
ENV = {**os.environ, "PYTHONPATH": f"{REPO}:{REPO}/scripts"}
ENV.pop("OPENROUTER_API_KEY", None)

AB_SCRIPT = f"{REPO}/scripts/run_blind_ab_3tomes.py"
JUDGE_SCRIPT = f"{REPO}/scripts/eval_refusal_blind.py"

def run_step(cmd, env=None, timeout=None):
    print("\n>>>", " ".join(cmd), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env or ENV)
    t0 = time.time()
    for line in p.stdout:
        print(line, end="", flush=True)
        if timeout and time.time() - t0 > timeout:
            p.kill(); raise RuntimeError(f"timeout ({timeout} s)")
    if p.wait() != 0:
        raise RuntimeError(f"Step failed: {' '.join(cmd)}")

def sync():
    for f in glob.glob(f"{W}/*.json*") + glob.glob(f"{W}/*.moipatch") + glob.glob(f"{W}/*.md"):
        shutil.copy2(f, OUTD)

# 1. Seleccionar/descargar hold-out, preparar splits disjuntos, censo y escanear target_base.xray.json (oculto a Brazo A)
if not os.path.exists(f"{W}/selected_holdout.json"):
    run_step([
        "python", "-u", AB_SCRIPT, "select-holdout",
        "--work-dir", W, "--base-dir", BASE,
        "--candidates", MODEL,
    ])
    sync()
SELECTED = json.load(open(f"{W}/selected_holdout.json"))
CENSUS = SELECTED["census"]
print("Hold-out bloqueado:", SELECTED["selected_model"], "| Censo base:", CENSUS)

In [ ]:
_CAPTURE_STUB = '''_EXCLUDED_BACKBONE_TOKENS = (
    "mtp", "visual", "vision", "vision_tower", "vision_model", "image_encoder",
    "mm_projector", "multi_modal_projector", "speculator", "dspark", "audio",
)
def _is_excluded_backbone_path(name: str) -> bool:
    low = name.lower()
    return any(tok in low for tok in _EXCLUDED_BACKBONE_TOKENS)
def _is_routed_expert_path(name: str) -> bool:
    return "experts" in name.lower().split(".")
'''

def build_standalone_moipatch(src_path, dst_path):
    src = open(src_path, encoding="utf-8").read()
    needle = "from .capture import _is_excluded_backbone_path, _is_routed_expert_path"
    assert needle in src, "Expected relative import missing in wsi/moipatch.py"
    open(dst_path, "w", encoding="utf-8").write(src.replace(needle, _CAPTURE_STUB, 1))

def run_isolated_arm(arm_name: str, mode: str, enable_wsi_mcp: bool):
    if os.path.exists(f"{W}/eval_{arm_name}.json"):
        print(f"Brazo {arm_name} ya evaluado; saltando.")
        return json.load(open(f"{W}/eval_{arm_name}.json"))

    !id -u agent >/dev/null 2>&1 || useradd -m -s /bin/bash agent
    shutil.rmtree(AG, ignore_errors=True)
    shutil.rmtree(HARNESS, ignore_errors=True)
    os.makedirs(f"{AG}/data"); os.makedirs(f"{AG}/out"); os.makedirs(HARNESS, exist_ok=True)

    for f in glob.glob(f"{W}/agent_*.jsonl"):
        shutil.copy2(f, f"{AG}/data")

    if mode == "raw_pytorch":
        build_standalone_moipatch(f"{REPO}/wsi/moipatch.py", f"{AG}/moipatch.py")
        shutil.copy2(f"{REPO}/benchmarks/blind_agent_open_phi4/TASK_RAW.md", f"{AG}/TASK.md")
    elif mode == "wsi_full":
        # Usa stage-workspace para copiar wsi/ + wsr/ + Atlas 3 Tomos + target_diagnosis.md con verificación 0-leakage
        run_step([
            "python", "-u", AB_SCRIPT, "stage-workspace",
            "--arm", "wsi_plus_atlas",
            "--agent-dir", AG, "--repo-dir", REPO, "--work-dir", W,
            "--time-limit-min", str(AGENT_MIN),
        ])
        # Sobrescribir TASK.md con el prompt abierto minimalista
        shutil.copy2(f"{REPO}/benchmarks/blind_agent_open_phi4/TASK_WSI_FULL.md", f"{AG}/TASK.md")
    else:
        raise ValueError(mode)

    for f in ("live_agent.py", "live_agent_repl.py"):
        shutil.copy2(f"{REPO}/scripts/{f}", HARNESS)

    !chown -R agent:agent {AG} && chmod -R a+rX {BASE} {HARNESS} && chmod 700 {REPO} {W} /root

    def as_agent(cmd):
        return subprocess.run(["runuser", "-u", "agent", "--", "bash", "-lc", cmd], capture_output=True, text=True)

    assert as_agent("python -c 'import torch; print(torch.cuda.is_available())'").stdout.strip() == "True"
    assert as_agent(f"ls {BASE}/config.json").returncode == 0
    if mode == "raw_pytorch":
        assert as_agent(f"PYTHONPATH={AG} python -c 'import moipatch'").returncode == 0
        assert as_agent(f"PYTHONPATH={AG} python -c 'import wsi'").returncode != 0, "wsi NO debe existir en raw_pytorch"
        assert as_agent(f"ls {AG}/atlas").returncode != 0, "atlas NO debe existir en raw_pytorch"
    else:
        assert as_agent(f"PYTHONPATH={AG} python -c 'import wsi, wsi.surgery_telemetry, wsi.atlas'").returncode == 0
        assert as_agent(f"ls {AG}/atlas/target_diagnosis.md").returncode == 0

    remount = False
    if USE_DRIVE and as_agent("ls /content/drive/MyDrive").returncode == 0:
        drive.flush_and_unmount(); remount = True
    for p in ("/content/drive/MyDrive", REPO, W, "/root"):
        assert as_agent(f"ls {p}").returncode != 0, f"agent PUEDE leer {p}"
    print(f"Sandbox OK para {arm_name} ({mode})")

    if not os.path.exists(f"{W}/{arm_name}.moipatch"):
        env = {
            **ENV,
            "PYTHONPATH": f"{AG}:{HARNESS}",
            "OPENROUTER_API_KEY": KEY,
            "HF_HUB_OFFLINE": "1",
            "TRANSFORMERS_OFFLINE": "1",
            "HOME": "/home/agent",
        }
        cmd = [
            "python", "-u", f"{HARNESS}/live_agent.py",
            "--model", AGENT_MODEL,
            "--workdir", AG,
            "--task", f"{AG}/TASK.md",
            "--log", f"{W}/{arm_name}_log.jsonl",
            "--summary", f"{W}/{arm_name}_run.json",
            "--time-limit-min", str(AGENT_MIN),
            "--run-as", "agent",
        ]
        if enable_wsi_mcp:
            cmd.append("--enable-wsi-mcp")
        try:
            run_step(cmd, env=env, timeout=(AGENT_MIN + 15) * 60)
        finally:
            !pkill -u agent || true
            del env
            if remount:
                drive.mount("/content/drive")
            for src, dst in ((f"{AG}/out/patch.moipatch", f"{W}/{arm_name}.moipatch"), (f"{AG}/out/notes.md", f"{W}/{arm_name}_notes.md")):
                if os.path.exists(src):
                    shutil.copy2(src, dst)
            sync()

    if os.path.exists(f"{W}/{arm_name}.moipatch"):
        run_step(["python", "-u", JUDGE_SCRIPT, "eval", "--work", W, "--model", BASE, "--patch", f"{W}/{arm_name}.moipatch", "--arm", arm_name])
        sync()
        return json.load(open(f"{W}/eval_{arm_name}.json"))
    ev = {"error": f"{arm_name} did not produce out/patch.moipatch"}
    open(f"{W}/eval_{arm_name}.json", "w").write(json.dumps(ev, indent=1))
    sync()
    return ev

In [ ]:
EV_RAW = run_isolated_arm("raw_pytorch", "raw_pytorch", enable_wsi_mcp=False)
EV_WSI = run_isolated_arm("wsi_full", "wsi_full", enable_wsi_mcp=True)

In [ ]:
RUN_RAW = json.load(open(f"{W}/raw_pytorch_run.json")) if os.path.exists(f"{W}/raw_pytorch_run.json") else {}
RUN_WSI = json.load(open(f"{W}/wsi_full_run.json")) if os.path.exists(f"{W}/wsi_full_run.json") else {}

print("| Arm | Turns | Wall min | Cost USD | MCP calls | Recovered | Toxic kept | Controls | Style OK | Success+Style | KB |")
print("| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: |")
for label, rj, ev in (("raw_pytorch", RUN_RAW, EV_RAW), ("wsi_full", RUN_WSI, EV_WSI)):
    print(
        f"| `{label}` | {rj.get('turns')} | {rj.get('wall_min')} | {rj.get('cost')} | "
        f"{rj.get('mcp_calls', 0)} | {ev.get('recovered')} | {ev.get('toxic_kept')} | "
        f"{ev.get('controls_identical')} | {(ev.get('style') or {}).get('style_ok')} | "
        f"{ev.get('success_with_style')} | {ev.get('patch_kb')} |"
    )

final = {
    "selected_holdout": SELECTED,
    "census": CENSUS,
    "arms": {
        "raw_pytorch": {"agent_run": RUN_RAW, "eval": EV_RAW},
        "wsi_full": {"agent_run": RUN_WSI, "eval": EV_WSI},
    },
    "_final": {
        "complete": True,
        "model": MODEL,
        "agent_model": AGENT_MODEL,
        "prompt_style": "open_minimal",
        "commit": COMMIT,
        "notebook": "blind_agent_open_ab_phi4_colab.ipynb",
        "gpu": subprocess.check_output(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True).strip(),
        "saved_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    },
}
dst = f"{FINALES}/{RUN_NAME}.json"
with open(dst + ".tmp", "w", encoding="utf-8") as fh:
    json.dump(final, fh, indent=1, ensure_ascii=False)
os.replace(dst + ".tmp", dst)
print("\nGuardado final en:", dst)
if not USE_DRIVE:
    try:
        from google.colab import files
        files.download(dst)
    except Exception:
        pass
print("<<<COMPACT_JSON_BEGIN>>>")
print(json.dumps(final, ensure_ascii=False))
print("<<<COMPACT_JSON_END>>>")